In [ ]:
# Section 1: Import Required Libraries
import os
import json
from zipfile import ZipFile
from xml.etree import ElementTree as ET
from pathlib import Path

print('Libraries loaded')

# Section 2: Load BRD from DOCX
brd_path = Path(r'C:\Prakash Chinnadurai_2022\Chellam Matrimony Project\Chellam Matrimony-AI Powered.docx')
assert brd_path.exists(), f'BRD not found: {brd_path}'

def read_docx_txt(path: Path) -> str:
    with ZipFile(path) as z:
        xml = z.read('word/document.xml')
    root = ET.fromstring(xml)
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    paragraphs = []
    for para in root.findall('.//w:p', ns):
        texts = [node.text or '' for node in para.findall('.//w:t', ns)]
        if texts:
            paragraphs.append(''.join(texts))
    return '\n'.join(paragraphs)

brd_text = read_docx_txt(brd_path)
print('BRD text loaded; length:', len(brd_text))

# Section 3: Extract Features from BRD

def extract_brd_features(text: str) -> list[dict]:
    features = []
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    current = None
    for line in lines:
        if line.lower().startswith('feature') or line.lower().startswith('requirement') or line.endswith(':'):
            if current:
                features.append(current)
            current = {'title': line.rstrip(':'), 'description': ''}
        elif current is not None:
            current['description'] += (' ' + line)
    if current:
        features.append(current)
    return features

brd_features = extract_brd_features(brd_text)
print('Extracted BRD features:', len(brd_features))
print(brd_features[:20])

# Section 4: Load Built Features List
built_features = [
    {'title': 'Profile completion and onboarding', 'status': 'built'},
    {'title': 'Search with filters', 'status': 'built'},
    {'title': 'Profile detail page from DB', 'status': 'built'},
    {'title': 'Interests and shortlist realtime updates', 'status': 'built'},
    {'title': 'Messaging realtime listener', 'status': 'built'},
    {'title': 'Admin route protection', 'status': 'built'},
    {'title': 'Membership payment page', 'status': 'partial'},
    {'title': 'AI matching integration', 'status': 'partial'},
    {'title': 'Notifications and profile views', 'status': 'partial'},
]

# Section 5: Compare BRD Features with Built Features

def compare_features(brd, built):
    mapped = []
    built_titles = {f['title'].lower(): f for f in built}
    for feature in brd:
        key = feature['title'].lower()
        built_match = built_titles.get(key)
        mapped.append({
            'brd_title': feature['title'],
            'description': feature['description'],
            'built_status': built_match['status'] if built_match else 'missing',
        })
    return mapped

comparison = compare_features(brd_features, built_features)
print('Comparison sample:', comparison[:20])

# Section 6: Identify Pending Features
pending = [item for item in comparison if item['built_status'] != 'built']
print('Pending features count:', len(pending))
print(pending)

# Section 7: Generate Summary Report
report = {
    'built': [item for item in comparison if item['built_status'] == 'built'],
    'partial': [item for item in comparison if item['built_status'] == 'partial'],
    'missing': [item for item in comparison if item['built_status'] == 'missing'],
}
print('Built features:', len(report['built']))
print('Partial features:', len(report['partial']))
print('Missing features:', len(report['missing']))
report
